In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random as rd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_curve, roc_auc_score, precision_recall_curve, average_precision_score

In [ ]:
def mse_loss(Xb, y, w):
    y_hat = predict(Xb, w)
    return np.mean((y_hat - y) ** 2)


def mse_grad(Xb, y, w):
    n = len(y)

    return (2/n) * (Xb.T @ (Xb @ w - y))




def predict(Xb, w):
    y_hat = Xb @ w
    return y_hat

In [ ]:
def add_bias_column(X):
    n = X.shape[0]

    ones = np.ones((n, 1), dtype=X.dtype)

    Xb = np.hstack([ones, X])

    return Xb

def main():
    df = pd.read_csv('Chevy_US_Data.csv')


    # For Classification

    df['Sales_Increase'] = (df['YOY Change'] > 0).astype(int) # Here we are predicting which sales increased or decreased


    SEED = rd.randrange(33, 42)



    #TARGET_COL = "Total Sales" # For Linear Regression
    TARGET_COL = 'Sales_Increase' # For Logistic Regression


    test_size = 0.15
    val_size = 0.15


    y = df[TARGET_COL].to_numpy(dtype=np.float64)



    X_df = df.drop(columns=[TARGET_COL])


    X_train, X_test, y_train, y_test = train_test_split( X_df, y, test_size=test_size, random_state=SEED)



    val_fraction = val_size / (1.0 - test_size)


    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=val_fraction, random_state=SEED)



    from pandas.api.types import is_numeric_dtype


    numeric_features = [c for c in X_df.columns if is_numeric_dtype(X_df[c])]

    categorical_features = [c for c in X_df.columns if c not in numeric_features]

    num_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])





    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])




    pre = ColumnTransformer(transformers=[
        ("num", num_pipe, numeric_features),
        ("cat", cat_pipe, categorical_features)
    ],
        remainder="drop",)





    # The following is for a Logistic Regression Model



    model = Pipeline(steps=[
        ('pre', pre),
        ('logreg', LogisticRegression(C=0.1, max_iter=2000, class_weight='balanced'))
    ])

    param_grid = {
        'logreg__C': [0.01, 0.1, 1.0, 10.0, 100.0]
    }

    grid = GridSearchCV(model, param_grid, scoring='accuracy', cv=5)

    grid.fit(X_train, y_train)


    model.set_params(logreg__C=grid.best_params_['logreg__C'])

    model.fit(X_train, y_train)


    y_train_p = model.predict(X_train)

    print("Train Accuracy: ", accuracy_score(y_train, y_train_p))
    print("Train Precision: ", precision_score(y_train, y_train_p))
    print("Train Recall: ", recall_score(y_train, y_train_p, zero_division=1))
    print("Train F1: ", f1_score(y_train, y_train_p))
    print("Train Confusion Matrix: ", confusion_matrix(y_train, y_train_p))
    print("Train Classification Report: ", classification_report(y_train, y_train_p))



    y_val_p = model.predict(X_val)



    print("Val Accuracy: ", accuracy_score(y_val, y_val_p))
    print("Val Precision: ", precision_score(y_val, y_val_p))
    print("Val Recall: ", recall_score(y_val, y_val_p, zero_division=1))
    print("Val F1: ", f1_score(y_val, y_val_p))
    print("Val Confusion Matrix: ", confusion_matrix(y_val, y_val_p))
    print("Val Classification Report: ", classification_report(y_val, y_val_p))




    y_val_proba = model.predict_proba(X_val)[:, 1]



    fpr, tpr, thresholds = roc_curve(y_val, y_val_proba)
    roc_auc = roc_auc_score(y_val, y_val_proba)



    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0,1],[0,1], linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f"ROC Curve (AUC = {roc_auc:.3f}")
    plt.show()



    threshold = 0.3


    y_val_pred_custom = (y_val_proba >= threshold).astype(int)


    v_accuracy = accuracy_score(y_val, y_val_pred_custom)
    v_precision = precision_score(y_val, y_val_pred_custom)
    v_recall = recall_score(y_val, y_val_pred_custom, zero_division=1)
    v_f1 = f1_score(y_val, y_val_pred_custom)
    v_cm = confusion_matrix(y_val, y_val_pred_custom)


    v_fpr, v_tpr, v_thresholds = roc_curve(y_val, y_val_proba)
    v_roc_auc = roc_auc_score(y_val, y_val_proba)



    print("T=0.3 Accuracy: ", v_accuracy)
    print("T=0.3 Precision: ", v_precision)
    print("T=0.3 Recall: ", v_recall)
    print("T=0.3 F1: ", v_f1)
    print("T=0.3 Confusion Matrix: ", v_cm)



    plt.figure()
    plt.plot(v_fpr, v_tpr)
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f"ROC Curve (AUC = {v_roc_auc:.3f}")
    plt.show()


    threshold = 0.5


    y_val_pred_custom = (y_val_proba >= threshold).astype(int)


    v_accuracy = accuracy_score(y_val, y_val_pred_custom)
    v_precision = precision_score(y_val, y_val_pred_custom)
    v_recall = recall_score(y_val, y_val_pred_custom, zero_division=1)
    v_f1 = f1_score(y_val, y_val_pred_custom)
    v_cm = confusion_matrix(y_val, y_val_pred_custom)


    v_fpr, v_tpr, v_thresholds = roc_curve(y_val_pred_custom, y_val_proba)
    v_roc_auc = roc_auc_score(y_val_pred_custom, y_val_proba)



    print("T=0.5 Accuracy: ", v_accuracy)
    print("T=0.5 Precision: ", v_precision)
    print("T=0.5 Recall: ", v_recall)
    print("T=0.5 F1: ", v_f1)
    print("T=0.5 Confusion Matrix: ", v_cm)



    plt.figure()
    plt.plot(v_fpr, v_tpr)
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f"ROC Curve (AUC = {v_roc_auc:.3f}")
    plt.show()



    threshold = 0.7


    y_val_pred_custom = (y_val_proba >= threshold).astype(int)


    v_accuracy = accuracy_score(y_val, y_val_pred_custom)
    v_precision = precision_score(y_val, y_val_pred_custom)
    v_recall = recall_score(y_val, y_val_pred_custom, zero_division=1)
    v_f1 = f1_score(y_val, y_val_pred_custom)
    v_cm = confusion_matrix(y_val, y_val_pred_custom)


    v_fpr, v_tpr, v_thresholds = roc_curve(y_val_pred_custom, y_val_proba)
    v_roc_auc = roc_auc_score(y_val_pred_custom, y_val_proba)



    print("T=0.7 Accuracy: ", v_accuracy)
    print("T=0.7 Precision: ", v_precision)
    print("T=0.7 Recall: ", v_recall)
    print("T=0.7 F1: ", v_f1)
    print("T=0.7 Confusion Matrix: ", v_cm)



    plt.figure()
    plt.plot(v_fpr, v_tpr)
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f"ROC Curve (AUC = {v_roc_auc:.3f}")
    plt.show()




    # Now we do the Test Data at the very end with the best model



    y_test_p = model.predict(X_test)



    print("Test Accuracy: ", accuracy_score(y_test, y_test_p))
    print("Test Precision: ", precision_score(y_test, y_test_p))
    print("Test Recall: ", recall_score(y_test, y_test_p, zero_division=1))
    print("Test F1: ", f1_score(y_test, y_test_p))
    print("Test Confusion Matrix: ", confusion_matrix(y_test, y_test_p))
    print("Test Classification Report: ", classification_report(y_test, y_test_p))













    # The following is for a Linear Regression Model, using Ridge (L2) regularization



    '''full_pipeline = Pipeline(steps=[
        ("pre", pre),
        ("reg", Ridge(alpha=1.0))
    ])


    param_grid = {
        'reg__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
    }

    grid = GridSearchCV(full_pipeline, param_grid, scoring='neg_mean_squared_error', cv=5)


    grid.fit(X_train, y_train)


    print(grid.best_params_)


    full_pipeline.set_params(reg__alpha=grid.best_params_['reg__alpha'])


    full_pipeline.fit(X_train, y_train)


    y_train_p = full_pipeline.predict(X_train)


    print("Train MSE: ", mean_squared_error(y_train, y_train_p))


    y_val_p = full_pipeline.predict(X_val)



    print("Validation MSE: ", mean_squared_error(y_val, y_val_p))'''








main()
